# Phase 5 v3.0: Hyperparameter Tuning - Random Forest

## Objective

**Best Model from Phase 4 (v3 with drop_first=False):** Random Forest

**Phase 4 v3 Results (18 cases, no data leakage):**
- **F1-Score**: 0.7347
- **Recall**: 0.9231 (92.31%) - Detected 72/78 attacks
- **Precision**: 0.6102 (61.02%)
- **FNR**: 0.0769 (7.69% of attacks missed)
- **FPR**: 0.0010 (0.102% false alarm rate)

**Practical Performance:**
- Test set: 45,010 records (78 timestomped)
- Detected: 72/78 attacks
- False alarms: 46
- Analyst workload: 0.64 false alarms per real attack
- Workload reduction: 99.7% (45K events → 118 flagged events)

**Current Hyperparameters (Phase 4 baseline):**
```python 
RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=42
)
```

---

## Phase 5 Goal

Optimize hyperparameters using **GridSearchCV** with:
- **Case-aware cross-validation** (GroupKFold) - No data leakage
- **F1-score optimization** - Best balance for forensic use case
- **Comprehensive search space** - 324 hyperparameter combinations

**Target**: Improve F1-score while maintaining high recall (>90%) and manageable false alarm rate

**Search Space:**
- `n_estimators`: [100, 200, 300]
- `max_depth`: [10, 15, 20, None]
- `min_samples_split`: [5, 10, 20]
- `min_samples_leaf`: [2, 5, 10]
- `max_features`: ['sqrt', 'log2', None]

**Total combinations**: 3 × 4 × 3 × 3 × 3 = **324 combinations**

---

## 1. Setup & Load Data

In [128]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import time

from sklearn.model_selection import GroupKFold, GridSearchCV, GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    make_scorer, f1_score, precision_score, recall_score, accuracy_score,
    matthews_corrcoef, cohen_kappa_score, balanced_accuracy_score,
    average_precision_score, roc_auc_score, confusion_matrix
)

warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (16, 10)

print("Libraries imported successfully")

Libraries imported successfully


In [129]:
# Define paths
BASE_DIR = Path('/Users/soni/Github/Digital-Detectives_Thesis')
INPUT_DIR = BASE_DIR / 'data' / 'processed' / 'Phase 3 - V2 Feature Selection'
OUTPUT_DIR = BASE_DIR / 'data' / 'processed' / 'Phase 5 - V2 Hyperparameter Tuning'

# Create output directory if it doesn't exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Model version for retraining with drop_first=False
MODEL_VERSION = "v3"  # v2 used drop_first=True, v3 uses drop_first=False for booleans

print("Directory Configuration:")
print(f"  Input: {INPUT_DIR}")
print(f"  Output: {OUTPUT_DIR}")
print(f"  Model Version: {MODEL_VERSION} (with drop_first=False for boolean columns)")

Directory Configuration:
  Input: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 3 - V2 Feature Selection
  Output: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 5 - V2 Hyperparameter Tuning
  Model Version: v3 (with drop_first=False for boolean columns)


In [130]:
# Load Phase 3 optimized dataset
print("Loading Phase 3 optimized dataset...")
input_file = INPUT_DIR / 'all_cases_combined_v2_phase3_final.csv'

# Force case_id to string to prevent pandas from auto-converting to int
df = pd.read_csv(input_file, encoding='utf-8-sig', dtype={'case_id': str})

print(f"\nDataset loaded:")
print(f"  Records: {len(df):,}")
print(f"  Cases: {df['case_id'].nunique()} (type: {df['case_id'].dtype})")
print(f"  Timestomped: {(df['timestomped'] == 1).sum():,} ({(df['timestomped'] == 1).sum() / len(df) * 100:.4f}%)")
print(f"  Normal: {(df['timestomped'] == 0).sum():,} ({(df['timestomped'] == 0).sum() / len(df) * 100:.4f}%)")

Loading Phase 3 optimized dataset...

Dataset loaded:
  Records: 283,118
  Cases: 18 (type: object)
  Timestomped: 280 (0.0989%)
  Normal: 282,838 (99.9011%)


---
## 2. Data Preprocessing

In [131]:
print("=" * 80)
print("DATA PREPROCESSING")
print("=" * 80)

identifier_cols = ['case_id', 'eventtime', 'eventtime_dt', 'filename', 'filepath', 'merge_key']
target_col = 'timestomped'
feature_cols = [col for col in df.columns if col not in identifier_cols + [target_col]]

X = df[feature_cols].copy()
y = df[target_col].copy()
groups = df['case_id'].astype(str).copy()  # For case-aware split

print(f"\nFeatures: {len(feature_cols)}")
print(f"Samples:  {len(X):,}")
print(f"Cases:    {groups.nunique()}")

# Convert boolean to int
bool_cols = X.select_dtypes(include=['bool']).columns.tolist()
for col in bool_cols:
    X[col] = X[col].astype(int)

# Handle object columns
object_cols = X.select_dtypes(include=['object']).columns.tolist()
cols_to_drop = [col for col in object_cols if X[col].nunique() > 1000]
cols_to_encode = [col for col in object_cols if X[col].nunique() <= 1000]

X = X.drop(columns=cols_to_drop) if cols_to_drop else X

# Boolean-like columns - fill NaN with False (not 'missing')
# LOGIC: If no LogFile data exists, these flags should be False (not "missing")
boolean_like_cols = ['copied_from_file', 'creation_time_changed_to_past', 'modified_time_changed_to_past', 
                     'accessed_time_changed_to_past', 'mft_modified_time_changed_to_past']

for col in cols_to_encode:
    if col not in X.columns:
        continue
    
    # CRITICAL FIX: Fill NaN with False for boolean columns (matching Phase 4 correction)
    if col in boolean_like_cols:
        X[col] = X[col].fillna(False)
        X[col] = X[col].astype(str)
    else:
        X[col] = X[col].fillna('missing')
    
    if X[col].nunique() < 5:
        # Use drop_first=True for boolean-like columns (only creates _True column)
        if col in boolean_like_cols:
            dummies = pd.get_dummies(X[col], prefix=col, drop_first=True)
            print(f"  One-hot encoding {col}: {X[col].nunique()} values -> {len(dummies.columns)} columns (drop_first=True)")
        else:
            dummies = pd.get_dummies(X[col], prefix=col, drop_first=True)
        X = pd.concat([X.drop(columns=[col]), dummies], axis=1)
    else:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])

X = X.fillna(-1)

print(f"\nPreprocessed shape: X={X.shape}, y={y.shape}")

DATA PREPROCESSING

Features: 50
Samples:  283,118
Cases:    18
  One-hot encoding copied_from_file: 2 values -> 1 columns (drop_first=True)
  One-hot encoding creation_time_changed_to_past: 2 values -> 1 columns (drop_first=True)
  One-hot encoding modified_time_changed_to_past: 2 values -> 1 columns (drop_first=True)
  One-hot encoding accessed_time_changed_to_past: 2 values -> 1 columns (drop_first=True)
  One-hot encoding mft_modified_time_changed_to_past: 2 values -> 1 columns (drop_first=True)

Preprocessed shape: X=(283118, 50), y=(283118,)


---
## 3. Case-Aware Train/Test Split

In [132]:
print("=" * 80)
print("CASE-AWARE TRAIN/TEST SPLIT")
print("=" * 80)

# NOTE: Phase 3 v3 already standardized case_ids to strings, but we keep this as
# defensive programming to ensure consistency across different data sources
print("\n🔧 Verifying case_id format...")
print(f"Unique cases: {df['case_id'].nunique()}")
print(f"Case_id type: {df['case_id'].dtype}")

# Ensure case_ids are strings (should already be from Phase 3, but being defensive)
df_reset = df.copy()
df_reset['case_id'] = df_reset['case_id'].astype(str).str.strip()

print(f"\nAfter verification: {df_reset['case_id'].nunique()} unique cases")
print(f"Cases: {sorted(df_reset['case_id'].unique())}")

# Reset indices to ensure alignment
df_reset = df_reset.reset_index(drop=True)
X_reset = X.reset_index(drop=True)
y_reset = y.reset_index(drop=True)

# Get case IDs and their labels
unique_cases = df_reset[['case_id', 'timestomped']].groupby('case_id')['timestomped'].max().reset_index()
print(f"\nTotal unique cases for split: {len(unique_cases)}")
print(f"Cases with timestomping: {(unique_cases['timestomped'] == 1).sum()}")
print(f"Cases without timestomping: {(unique_cases['timestomped'] == 0).sum()}")

# Split cases (same as Phase 4 for fair comparison)
from sklearn.model_selection import train_test_split

train_cases, test_cases = train_test_split(
    unique_cases['case_id'].values,
    test_size=0.2,
    random_state=42,
    stratify=unique_cases['timestomped'].values
)

print(f"\nTrain cases: {len(train_cases)}")
print(f"  {sorted(train_cases)}")
print(f"\nTest cases: {len(test_cases)}")
print(f"  {sorted(test_cases)}")

# Create train and test masks
train_mask = df_reset['case_id'].isin(train_cases)
test_mask = df_reset['case_id'].isin(test_cases)

# Split data using aligned indices
X_train = X_reset[train_mask].copy()
X_test = X_reset[test_mask].copy()
y_train = y_reset[train_mask].copy()
y_test = y_reset[test_mask].copy()

# Create groups from the reset df
groups_train = df_reset.loc[train_mask, 'case_id'].astype(str)
groups_test = df_reset.loc[test_mask, 'case_id'].astype(str)

print(f"\nTrain:")
print(f"  Records: {len(X_train):,}")
print(f"  Timestomped: {(y_train == 1).sum():,} ({(y_train == 1).sum() / len(y_train) * 100:.4f}%)")
print(f"  Cases: {groups_train.nunique()}")

print(f"\nTest (held out for final evaluation):")
print(f"  Records: {len(X_test):,}")
print(f"  Timestomped: {(y_test == 1).sum():,} ({(y_test == 1).sum() / len(y_test) * 100:.4f}%)")
print(f"  Cases: {groups_test.nunique()}")

# Verify no overlap
overlap = set(groups_train.unique()) & set(groups_test.unique())
if not overlap:
    print(f"\n✓ No case overlap between train and test")
else:
    print(f"\n⚠️ ERROR: Cases overlap: {overlap}")
    raise ValueError(f"Data leakage detected! Cases {overlap} appear in both train and test sets.")

CASE-AWARE TRAIN/TEST SPLIT

🔧 Verifying case_id format...
Unique cases: 18
Case_id type: object

After verification: 18 unique cases
Cases: ['01-APT17', '02-APT19', '04-APT28', '05-APT29', '1', '10', '10-DarkHotel663', '11', '11-DarkHotelbbd', '12', '2', '3', '4', '5', '6', '7', '8', '9']

Total unique cases for split: 18
Cases with timestomping: 18
Cases without timestomping: 0

Train cases: 14
  ['01-APT17', '02-APT19', '04-APT28', '05-APT29', '1', '10', '11-DarkHotelbbd', '12', '3', '4', '5', '7', '8', '9']

Test cases: 4
  ['10-DarkHotel663', '11', '2', '6']

Train:
  Records: 238,108
  Timestomped: 202 (0.0848%)
  Cases: 14

Test (held out for final evaluation):
  Records: 45,010
  Timestomped: 78 (0.1733%)
  Cases: 4

✓ No case overlap between train and test


---
## 4. Baseline Model (Phase 4 Results)

In [133]:
print("=" * 80)
print("BASELINE MODEL (PHASE 4 HYPERPARAMETERS)")
print("=" * 80)

baseline_params = {
    'n_estimators': 100,
    'max_depth': 10,
    'min_samples_split': 10,
    'min_samples_leaf': 5,
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs': -1
}

print("\nBaseline hyperparameters:")
for param, value in baseline_params.items():
    print(f"  {param:<20s}: {value}")

# Train baseline
print("\nTraining baseline model...")
baseline_model = RandomForestClassifier(**baseline_params)
baseline_model.fit(X_train, y_train)

# Evaluate baseline
y_pred_baseline = baseline_model.predict(X_test)
y_pred_proba_baseline = baseline_model.predict_proba(X_test)[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred_baseline)
precision = precision_score(y_test, y_pred_baseline, zero_division=0)
recall = recall_score(y_test, y_pred_baseline, zero_division=0)
f1 = f1_score(y_test, y_pred_baseline, zero_division=0)
auc_roc = roc_auc_score(y_test, y_pred_proba_baseline)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_baseline)
tn, fp, fn, tp = cm.ravel()

# Calculate FNR and FPR
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

# Store results
baseline_results = {
    'f1': f1,
    'recall': recall,
    'precision': precision,
    'fnr': fnr,
    'fpr': fpr,
    'accuracy': accuracy,
    'auc_roc': auc_roc,
    'tp': tp,
    'tn': tn,
    'fp': fp,
    'fn': fn
}

print("\n" + "=" * 80)
print("BASELINE FORENSIC PERFORMANCE")
print("=" * 80)

print(f"\nPrimary Metrics:")
print(f"  F1-Score:  {f1:.4f}")
print(f"  Recall:    {recall:.4f} ({recall*100:.2f}%)")
print(f"  Precision: {precision:.4f} ({precision*100:.2f}%)")

print(f"\nError Analysis:")
print(f"  FNR: {fnr:.4f} ({fnr*100:.2f}%) - Missed {fn}/{fn+tp} attacks")
print(f"  FPR: {fpr:.4f} ({fpr*100:.2f}%) - {fp} false alarms")

print(f"\nConfusion Matrix:")
print(f"  TN: {tn:6,}  FP: {fp:6,}")
print(f"  FN: {fn:6,}  TP: {tp:6,}")

print(f"\nPractical Summary:")
print(f"  • Detected {tp}/{tp+fn} attacks ({recall*100:.1f}%)")
print(f"  • {fp} false alarms ({fpr*100:.3f}% of normal events)")
if tp > 0:
    workload_ratio = fp / tp
    print(f"  • Analyst workload: {workload_ratio:.2f} false alarms per real attack")

print(f"\nReference Metrics:")
print(f"  Accuracy: {accuracy:.4f} (less meaningful with imbalance)")
print(f"  AUC-ROC:  {auc_roc:.4f} (good for comparison)")

print("\n" + "=" * 80)

BASELINE MODEL (PHASE 4 HYPERPARAMETERS)

Baseline hyperparameters:
  n_estimators        : 100
  max_depth           : 10
  min_samples_split   : 10
  min_samples_leaf    : 5
  class_weight        : balanced
  random_state        : 42
  n_jobs              : -1

Training baseline model...

BASELINE FORENSIC PERFORMANCE

Primary Metrics:
  F1-Score:  0.6025
  Recall:    0.9231 (92.31%)
  Precision: 0.4472 (44.72%)

Error Analysis:
  FNR: 0.0769 (7.69%) - Missed 6/78 attacks
  FPR: 0.0020 (0.20%) - 89 false alarms

Confusion Matrix:
  TN: 44,843  FP:     89
  FN:      6  TP:     72

Practical Summary:
  • Detected 72/78 attacks (92.3%)
  • 89 false alarms (0.198% of normal events)
  • Analyst workload: 1.24 false alarms per real attack

Reference Metrics:
  Accuracy: 0.9979 (less meaningful with imbalance)
  AUC-ROC:  0.9993 (good for comparison)



---
## 5. Hyperparameter Grid Search

### Search Space:
- **n_estimators**: Number of trees [100, 200, 300]
- **max_depth**: Maximum tree depth [10, 15, 20, None]
- **min_samples_split**: Min samples to split [5, 10, 20]
- **min_samples_leaf**: Min samples in leaf [2, 5, 10]
- **max_features**: Features per split ['sqrt', 'log2', None]

**Total combinations**: 3 × 4 × 3 × 3 × 3 = 324 combinations

**Cross-validation**: 5-fold GroupKFold (case-aware)

In [134]:
print("=" * 80)
print("HYPERPARAMETER GRID SEARCH")
print("=" * 80)

# Define parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [5, 10, 20],
    'min_samples_leaf': [2, 5, 10],
    'max_features': ['sqrt', 'log2', None]
}

print("\nParameter grid:")
for param, values in param_grid.items():
    print(f"  {param:<20s}: {values}")

total_combinations = np.prod([len(v) for v in param_grid.values()])
print(f"\nTotal combinations: {total_combinations}")

# Case-aware cross-validation
cv = GroupKFold(n_splits=5)
print(f"Cross-validation: {cv.n_splits}-fold GroupKFold (case-aware)")

# Custom scorer for F1 (optimizing for minority class)
f1_scorer = make_scorer(f1_score, zero_division=0)

# GridSearchCV
grid_search = GridSearchCV(
    estimator=RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
    param_grid=param_grid,
    scoring=f1_scorer,
    cv=cv,
    verbose=2,
    n_jobs=-1,
    return_train_score=True
)

print(f"\nStarting grid search...")
print(f"This will take approximately {total_combinations * 5 / 60:.0f}-{total_combinations * 5 / 30:.0f} minutes...\n")

start_time = time.time()
grid_search.fit(X_train, y_train, groups=groups_train)
elapsed_time = time.time() - start_time

print(f"\n✓ Grid search completed in {elapsed_time / 60:.1f} minutes")

HYPERPARAMETER GRID SEARCH

Parameter grid:
  n_estimators        : [100, 200, 300]
  max_depth           : [10, 15, 20, None]
  min_samples_split   : [5, 10, 20]
  min_samples_leaf    : [2, 5, 10]
  max_features        : ['sqrt', 'log2', None]

Total combinations: 324
Cross-validation: 5-fold GroupKFold (case-aware)

Starting grid search...
This will take approximately 27-54 minutes...

Fitting 5 folds for each of 324 candidates, totalling 1620 fits
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   7.0s
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   6.9s
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   7.6s
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   7.3s
[CV] END max_depth=10, max_features=sqrt, min_samples_leaf=2, min

---
## 6. Best Hyperparameters

In [135]:
print("=" * 80)
print("BEST HYPERPARAMETERS")
print("=" * 80)

print(f"\nBest cross-validation F1-score: {grid_search.best_score_:.4f}")

print("\nBest hyperparameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param:<20s}: {value}")

print("\n" + "=" * 80)
print("COMPARISON: BASELINE vs BEST")
print("=" * 80)

print("\nBaseline (Phase 4):")
for param in grid_search.best_params_.keys():
    baseline_value = baseline_params.get(param, 'N/A')
    print(f"  {param:<20s}: {baseline_value}")

print("\nOptimized (Phase 5):")
for param, value in grid_search.best_params_.items():
    print(f"  {param:<20s}: {value}")

BEST HYPERPARAMETERS

Best cross-validation F1-score: 0.4084

Best hyperparameters:
  max_depth           : 10
  max_features        : sqrt
  min_samples_leaf    : 2
  min_samples_split   : 20
  n_estimators        : 100

COMPARISON: BASELINE vs BEST

Baseline (Phase 4):
  max_depth           : 10
  max_features        : N/A
  min_samples_leaf    : 5
  min_samples_split   : 10
  n_estimators        : 100

Optimized (Phase 5):
  max_depth           : 10
  max_features        : sqrt
  min_samples_leaf    : 2
  min_samples_split   : 20
  n_estimators        : 100


---
## 7. Top 10 Parameter Combinations

In [136]:
print("=" * 80)
print("TOP 10 PARAMETER COMBINATIONS")
print("=" * 80)

# Get results
cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results = cv_results.sort_values('rank_test_score')

print("\nTop 10 combinations by F1-score:")
print(f"{'Rank':<6} {'Mean F1':<10} {'Std F1':<10} {'Parameters':<60}")
print("=" * 80)

for idx, row in cv_results.head(10).iterrows():
    rank = int(row['rank_test_score'])
    mean_score = row['mean_test_score']
    std_score = row['std_test_score']
    params = row['params']
    params_str = ', '.join([f"{k}={v}" for k, v in params.items()])
    print(f"{rank:<6} {mean_score:<10.4f} {std_score:<10.4f} {params_str[:60]}")

# Save full results
cv_results_file = OUTPUT_DIR / 'grid_search_results_v2.csv'
cv_results.to_csv(cv_results_file, index=False)
print(f"\n✓ Saved full results to: {cv_results_file}")

TOP 10 PARAMETER COMBINATIONS

Top 10 combinations by F1-score:
Rank   Mean F1    Std F1     Parameters                                                  
1      0.4084     0.2236     max_depth=10, max_features=sqrt, min_samples_leaf=2, min_sam
2      0.4080     0.2350     max_depth=10, max_features=log2, min_samples_leaf=5, min_sam
2      0.4080     0.2350     max_depth=10, max_features=log2, min_samples_leaf=5, min_sam
2      0.4080     0.2350     max_depth=10, max_features=log2, min_samples_leaf=5, min_sam
5      0.4071     0.2440     max_depth=10, max_features=log2, min_samples_leaf=5, min_sam
5      0.4071     0.2440     max_depth=10, max_features=log2, min_samples_leaf=5, min_sam
7      0.4070     0.2416     max_depth=10, max_features=log2, min_samples_leaf=2, min_sam
8      0.4047     0.2394     max_depth=10, max_features=log2, min_samples_leaf=5, min_sam
8      0.4047     0.2394     max_depth=10, max_features=log2, min_samples_leaf=5, min_sam
10     0.4012     0.2478     max_dep

---
## 8. Evaluate Tuned Model on Test Set

### Evaluation Focus

We prioritize **forensically-relevant metrics** over standard ML metrics:

**Primary Metrics (Forensic Importance):**
1. **Recall** - Detection rate (minimize missed attacks)
2. **Precision** - Investigation efficiency (minimize wasted effort)
3. **F1-Score** - Overall balance

**Secondary Metrics (Error Analysis):**
- **FNR** (False Negative Rate) - Security risk
- **FPR** (False Positive Rate) - Analyst workload

**Reference Metrics** (Less meaningful with extreme imbalance):
- Accuracy (~99.9% even for naive baseline)
- AUC-ROC (good for comparison, not interpretation)

### Test Set

- **Held-out cases**: 4 cases never seen during training
- **Case-aware split**: No data leakage
- **Class distribution**: 41 timestomped / 81,757 total events (0.05%)

In [137]:
print("=" * 80)
print("FINAL EVALUATION ON TEST SET")
print("=" * 80)

# Best model from grid search
best_model = grid_search.best_estimator_

# Predict on test set
y_pred_tuned = best_model.predict(X_test)
y_pred_proba_tuned = best_model.predict_proba(X_test)[:, 1]

# Calculate all metrics
accuracy = accuracy_score(y_test, y_pred_tuned)
precision = precision_score(y_test, y_pred_tuned, zero_division=0)
recall = recall_score(y_test, y_pred_tuned, zero_division=0)
f1 = f1_score(y_test, y_pred_tuned, zero_division=0)
auc_roc = roc_auc_score(y_test, y_pred_proba_tuned)
auc_pr = average_precision_score(y_test, y_pred_proba_tuned)
mcc = matthews_corrcoef(y_test, y_pred_tuned)
kappa = cohen_kappa_score(y_test, y_pred_tuned)
balanced_acc = balanced_accuracy_score(y_test, y_pred_tuned)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_tuned)
tn, fp, fn, tp = cm.ravel()

# Calculate FNR and FPR
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

# Store results
tuned_results = {
    'accuracy': accuracy,
    'precision': precision,
    'recall': recall,
    'f1': f1,
    'auc_roc': auc_roc,
    'auc_pr': auc_pr,
    'mcc': mcc,
    'kappa': kappa,
    'balanced_acc': balanced_acc,
    'fnr': fnr,
    'fpr': fpr,
    'tp': tp,
    'tn': tn,
    'fp': fp,
    'fn': fn
}

print("\n" + "=" * 80)
print("📊 FORENSIC PERFORMANCE METRICS")
print("=" * 80)

print(f"\nPrimary Metrics (Forensic Importance):")
print(f"  Recall (Detection Rate):       {recall:.4f} ({recall*100:.2f}%)")
print(f"  Precision (Flag Accuracy):     {precision:.4f} ({precision*100:.2f}%)")
print(f"  F1-Score (Overall Balance):    {f1:.4f}")

print(f"\nERROR ANALYSIS:")
print(f"  False Negative Rate:  {fnr:.4f} ({fnr*100:.2f}%) - Missed {fn}/{fn+tp} attacks")
print(f"  False Positive Rate:  {fpr:.4f} ({fpr*100:.2f}%) - {fp} false alarms per {fp+tn:,} normal events")

print(f"\nCONFUSION MATRIX:")
print(f"                  Predicted")
print(f"                  Normal    Attack")
print(f"  Actual Normal   {tn:6,}    {fp:6,}   ({tn/(tn+fp)*100:.2f}% correct)")
print(f"  Actual Attack   {fn:6,}    {tp:6,}   ({tp/(tp+fn)*100:.2f}% detected)")

print(f"\nPRACTICAL INTERPRETATION:")
total_attacks = tp + fn
detected = tp
missed = fn
false_alarms = fp

print(f"  • Detected {detected}/{total_attacks} attacks ({recall*100:.1f}%)")
print(f"  • Missed {missed}/{total_attacks} attacks ({fnr*100:.1f}%)")
print(f"  • Generated {false_alarms} false alarms ({fpr*100:.3f}% of {fp+tn:,} normal events)")

if fp > 0:
    ppv = tp / (tp + fp)
    workload_ratio = fp / tp if tp > 0 else 0
    print(f"  • Of {tp+fp} flagged events, {tp} were real attacks ({ppv*100:.1f}% precision)")
    print(f"  • Analyst investigates {workload_ratio:.2f} false alarms per real attack")

print(f"\nREFERENCE METRICS:")
print(f"  Accuracy:           {accuracy:.4f} (less meaningful with imbalance)")
print(f"  Balanced Accuracy:  {balanced_acc:.4f}")
print(f"  AUC-ROC:            {auc_roc:.4f}")
print(f"  AUC-PR:             {auc_pr:.4f}")
print(f"  MCC:                {mcc:.4f}")
print(f"  Cohen's Kappa:      {kappa:.4f}")

print("\n" + "=" * 80)
print("FORENSIC USE CASE ASSESSMENT")
print("=" * 80)

if recall >= 0.90:
    print(f"✓ EXCELLENT Detection: {recall*100:.1f}% detection rate")
elif recall >= 0.80:
    print(f"✓ VERY GOOD Detection: {recall*100:.1f}% detection rate")
elif recall >= 0.70:
    print(f"✓ GOOD Detection: {recall*100:.1f}% detection rate")
else:
    print(f"MODERATE Detection: {recall*100:.1f}% detection rate (improvement recommended)")

if precision >= 0.70:
    print(f"✓ LOW False Alarm Rate: {precision*100:.1f}% precision")
elif precision >= 0.50:
    print(f"MODERATE False Alarm Rate: {precision*100:.1f}% precision")
else:
    print(f"HIGH False Alarm Rate: {precision*100:.1f}% precision")

if fpr <= 0.001:
    print(f"✓ VERY LOW FPR: {fpr*100:.3f}% of normal events flagged")
elif fpr <= 0.01:
    print(f"✓ LOW FPR: {fpr*100:.3f}% of normal events flagged")
else:
    print(f"HIGH FPR: {fpr*100:.3f}% of normal events flagged")

workload_ratio = fp / tp if tp > 0 else 0
if workload_ratio <= 1.0:
    print(f"✓ EFFICIENT Workflow: {workload_ratio:.2f} false alarms per real attack")
elif workload_ratio <= 3.0:
    print(f"✓ ACCEPTABLE Workflow: {workload_ratio:.2f} false alarms per real attack")
else:
    print(f"HIGH Workload: {workload_ratio:.2f} false alarms per real attack")


FINAL EVALUATION ON TEST SET

📊 FORENSIC PERFORMANCE METRICS

Primary Metrics (Forensic Importance):
  Recall (Detection Rate):       0.9103 (91.03%)
  Precision (Flag Accuracy):     0.8554 (85.54%)
  F1-Score (Overall Balance):    0.8820

ERROR ANALYSIS:
  False Negative Rate:  0.0897 (8.97%) - Missed 7/78 attacks
  False Positive Rate:  0.0003 (0.03%) - 12 false alarms per 44,932 normal events

CONFUSION MATRIX:
                  Predicted
                  Normal    Attack
  Actual Normal   44,920        12   (99.97% correct)
  Actual Attack        7        71   (91.03% detected)

PRACTICAL INTERPRETATION:
  • Detected 71/78 attacks (91.0%)
  • Missed 7/78 attacks (9.0%)
  • Generated 12 false alarms (0.027% of 44,932 normal events)
  • Of 83 flagged events, 71 were real attacks (85.5% precision)
  • Analyst investigates 0.17 false alarms per real attack

REFERENCE METRICS:
  Accuracy:           0.9996 (less meaningful with imbalance)
  Balanced Accuracy:  0.9550
  AUC-ROC:         

---
## 9. Baseline vs Tuned Comparison

In [138]:
print("\n" + "=" * 80)
print("BASELINE vs TUNED MODEL COMPARISON")
print("=" * 80)

comparison = pd.DataFrame({
    'Metric': ['F1-Score', 'Recall', 'Precision', 'FNR', 'FPR'],
    'Baseline (Phase 4)': [
        baseline_results['f1'],
        baseline_results['recall'],
        baseline_results['precision'],
        baseline_results['fnr'],
        baseline_results['fpr']
    ],
    'Tuned (Phase 5)': [
        tuned_results['f1'],
        tuned_results['recall'],
        tuned_results['precision'],
        tuned_results['fnr'],
        tuned_results['fpr']
    ]
})

comparison['Improvement'] = comparison['Tuned (Phase 5)'] - comparison['Baseline (Phase 4)']
comparison['% Change'] = (comparison['Improvement'] / comparison['Baseline (Phase 4)'].abs() * 100).round(2)

# Adjust interpretation for FNR and FPR (lower is better)
comparison.loc[comparison['Metric'].isin(['FNR', 'FPR']), 'Interpretation'] = \
    comparison.loc[comparison['Metric'].isin(['FNR', 'FPR']), 'Improvement'].apply(
        lambda x: 'Better' if x < 0 else ('Worse' if x > 0 else 'Same')
    )
comparison.loc[~comparison['Metric'].isin(['FNR', 'FPR']), 'Interpretation'] = \
    comparison.loc[~comparison['Metric'].isin(['FNR', 'FPR']), 'Improvement'].apply(
        lambda x: 'Better' if x > 0 else ('Worse' if x < 0 else 'Same')
    )

print("\n📊 FORENSIC METRICS COMPARISON:")
print("=" * 80)
print(comparison.to_string(index=False))

# Save comparison
comparison_file = OUTPUT_DIR / f'baseline_vs_tuned_comparison_{MODEL_VERSION}.csv'
comparison.to_csv(comparison_file, index=False)
print(f"\n✓ Saved comparison to: {comparison_file}")

# Interpretation
print("\n" + "=" * 80)
print("INTERPRETATION:")
print("=" * 80)

f1_improvement = tuned_results['f1'] - baseline_results['f1']
recall_improvement = tuned_results['recall'] - baseline_results['recall']
precision_improvement = tuned_results['precision'] - baseline_results['precision']

if f1_improvement > 0.05:
    print(f"\n✓ SIGNIFICANT IMPROVEMENT: F1-Score improved by {f1_improvement:.4f} ({f1_improvement/baseline_results['f1']*100:.1f}%)")
elif f1_improvement > 0.01:
    print(f"\n✓ MODERATE IMPROVEMENT: F1-Score improved by {f1_improvement:.4f} ({f1_improvement/baseline_results['f1']*100:.1f}%)")
elif f1_improvement > 0:
    print(f"\n✓ MINOR IMPROVEMENT: F1-Score improved by {f1_improvement:.4f} ({f1_improvement/baseline_results['f1']*100:.1f}%)")
else:
    print(f"\n NO IMPROVEMENT: F1-Score changed by {f1_improvement:.4f}")

if recall_improvement > 0:
    print(f"✓ Detection improved: Now catching {tuned_results['recall']*100:.1f}% of attacks (was {baseline_results['recall']*100:.1f}%)")
else:
    print(f"Detection unchanged: Still catching {tuned_results['recall']*100:.1f}% of attacks")

if precision_improvement > 0:
    print(f"✓ Precision improved: {tuned_results['precision']*100:.1f}% of flags are real (was {baseline_results['precision']*100:.1f}%)")
else:
    print(f"Precision unchanged: {tuned_results['precision']*100:.1f}% of flags are real")

print("\nFORENSIC IMPACT:")
print(f"  • Detects {tuned_results['tp']}/{tuned_results['tp']+tuned_results['fn']} attacks on held-out cases")
print(f"  • Generates {tuned_results['fp']} false alarms per {tuned_results['tp']+tuned_results['fn']} real attacks")
print(f"  • Analyst workload: {tuned_results['fp']/tuned_results['tp']:.2f} false positives per true positive")

print("\n" + "=" * 80)


BASELINE vs TUNED MODEL COMPARISON

📊 FORENSIC METRICS COMPARISON:
   Metric  Baseline (Phase 4)  Tuned (Phase 5)  Improvement  % Change Interpretation
 F1-Score            0.602510         0.881988     0.279477     46.39         Better
   Recall            0.923077         0.910256    -0.012821     -1.39          Worse
Precision            0.447205         0.855422     0.408217     91.28         Better
      FNR            0.076923         0.089744     0.012821     16.67          Worse
      FPR            0.001981         0.000267    -0.001714    -86.52         Better

✓ Saved comparison to: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 5 - V2 Hyperparameter Tuning/baseline_vs_tuned_comparison_v3.csv

INTERPRETATION:

✓ SIGNIFICANT IMPROVEMENT: F1-Score improved by 0.2795 (46.4%)
Detection unchanged: Still catching 91.0% of attacks
✓ Precision improved: 85.5% of flags are real (was 44.7%)

FORENSIC IMPACT:
  • Detects 71/78 attacks on held-out cases
  • Generates

---
## 10. Feature Importances (Tuned Model)

In [139]:
print("=" * 80)
print("TOP 20 FEATURE IMPORTANCES (TUNED MODEL)")
print("=" * 80)

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\n{'Feature':<50s} {'Importance':>12s}")
print("=" * 80)

for idx, row in feature_importance.head(20).iterrows():
    print(f"{row['feature']:<50s} {row['importance']:>12.4f}")

# Save feature importances
feature_importance_file = OUTPUT_DIR / 'feature_importances_tuned_v2.csv'
feature_importance.to_csv(feature_importance_file, index=False)
print(f"\n✓ Saved to: {feature_importance_file}")

TOP 20 FEATURE IMPORTANCES (TUNED MODEL)

Feature                                              Importance
usn_usn                                                  0.1918
usn_event_info                                           0.1120
path_depth                                               0.0973
events_in_5min_window                                    0.0736
events_in_1min_window                                    0.0686
timestamp_manipulation_pattern_score                     0.0658
filename_length                                          0.0488
event_frequency_per_file                                 0.0468
usn_complete_manipulation_pattern                        0.0433
is_archive                                               0.0428
usn_file_closed                                          0.0371
event_frequency_per_case                                 0.0271
is_executable                                            0.0245
lf_creation_time_before                                  0.014

---
## 11. Save Tuned Model

In [140]:
import joblib

print("=" * 80)
print("SAVING TUNED MODEL")
print("=" * 80)

model_file = OUTPUT_DIR / 'random_forest_tuned_v2.pkl'
joblib.dump(best_model, model_file)

print(f"\n✓ Saved tuned model to: {model_file}")
print(f"\nModel details:")
print(f"  Best parameters: {grid_search.best_params_}")
print(f"  Test F1-Score: {tuned_results['f1']:.4f}")
print(f"  Test Recall: {tuned_results['recall']:.4f}")
print(f"  Test Precision: {tuned_results['precision']:.4f}")

SAVING TUNED MODEL

✓ Saved tuned model to: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 5 - V2 Hyperparameter Tuning/random_forest_tuned_v2.pkl

Model details:
  Best parameters: {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 20, 'n_estimators': 100}
  Test F1-Score: 0.8820
  Test Recall: 0.9103
  Test Precision: 0.8554
